# 08d — Cybersectony Legitimate Emails (English)

**C1 Source Type:** `Fichier de données (CSV / data file)`

---

## Objective

Load **legitimate (non-phishing) emails** from the `cybersectony/PhishingEmailDetectionv2.0`
dataset on HuggingFace Hub. We extract only the `label=0` (legitimate) subset to
complement the BigQuery phishing extraction (notebook 03) which focused on `label=1`.

### Dataset Info

| Field | Value |
|-------|-------|
| Source | [HuggingFace: cybersectony/PhishingEmailDetectionv2.0](https://huggingface.co/datasets/cybersectony/PhishingEmailDetectionv2.0) |
| Format | Parquet via HuggingFace `datasets` library |
| Total Rows | ~120K (full dataset) |
| Subset | Legitimate only (`label=0`) → ~6.8K rows |
| Language | English |

### Pipeline

```
HuggingFace Hub → Load with datasets → Filter label=0 → Normalize schema → Export CSV
```

### Output

- `data/raw/csv/en/cybersectony_legit_<N>_<date>.csv`

In [1]:
# ── Imports & Constants ──────────────────────────────────────────────
from __future__ import annotations

import hashlib
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from datasets import load_dataset

# ── Configuration ────────────────────────────────────────────────────
OUTPUT_DIR: Path = Path("data/raw/csv/en")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_TEXT_LENGTH: int = 20
SCHEMA_COLS: list[str] = ["text", "label", "source", "language"]

print(f"Output dir   : {OUTPUT_DIR.resolve()}")
print(f"Min length   : {MIN_TEXT_LENGTH}")

Output dir   : /Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/data/raw/csv/en
Min length   : 20


## 1. Load Dataset

In [2]:
# ── Load cybersectony/PhishingEmailDetectionv2.0 ──────────────────────
print("Loading cybersectony/PhishingEmailDetectionv2.0 from HuggingFace...")

ds = load_dataset("cybersectony/PhishingEmailDetectionv2.0", split="train")
df_full: pd.DataFrame = ds.to_pandas()  # pyright: ignore[reportAssignmentType]

print(f"Full dataset : {len(df_full):,} rows")
print(f"Columns      : {list(df_full.columns)}")
print(f"Label values : {df_full['label'].value_counts().to_dict()}")

# Filter for legitimate emails only (label=0)
df_legit: pd.DataFrame = df_full[df_full["label"] == 0].copy()
print(f"\nLegit subset : {len(df_legit):,} rows (label=0)")

Loading cybersectony/PhishingEmailDetectionv2.0 from HuggingFace...
Full dataset : 120,000 rows
Columns      : ['content', 'label']
Label values : {3: 53350, 2: 53157, 0: 6809, 1: 6684}

Legit subset : 6,809 rows (label=0)


## 2. Normalize Schema

In [3]:
# ── Normalize to unified schema ──────────────────────────────────────
text_col: str = "content" if "content" in df_legit.columns else "text"

df_norm = pd.DataFrame({
    "text": df_legit[text_col].astype(str),
    "label": "ham",
    "source": "cybersectony_phishing_v2",
    "language": "en",
})

# Filter short texts
before: int = len(df_norm)
df_norm = df_norm[df_norm["text"].str.len() >= MIN_TEXT_LENGTH].reset_index(drop=True)
print(f"Before filter : {before:,}")
print(f"After filter  : {len(df_norm):,} (removed {before - len(df_norm)} short texts)")

Before filter : 6,809
After filter  : 6,695 (removed 114 short texts)


## 3. Deduplicate

In [4]:
# ── Deduplication by text hash ────────────────────────────────────────
before = len(df_norm)
df_norm["text_hash"] = df_norm["text"].str[:300].apply(
    lambda t: hashlib.sha256(t.encode("utf-8", errors="ignore")).hexdigest()
)
df_norm = (
    df_norm.drop_duplicates(subset="text_hash", keep="first")
    .drop(columns="text_hash")
    .reset_index(drop=True)
)
after: int = len(df_norm)
print(f"Before dedup : {before:,}")
print(f"After dedup  : {after:,}")
print(f"Removed      : {before - after:,}")

Before dedup : 6,695
After dedup  : 6,606
Removed      : 89


## 4. Export

In [5]:
# ── Export ─────────────────────────────────────────────────────────────
timestamp: str = datetime.now(timezone.utc).strftime("%Y%m%d")
n_rows: int = len(df_norm)

if n_rows > 0:
    filename: str = f"cybersectony_legit_{n_rows}_{timestamp}.csv"
    output_path: Path = OUTPUT_DIR / filename
    df_norm.to_csv(output_path, index=False, encoding="utf-8")

    size_mb: float = output_path.stat().st_size / (1024 * 1024)
    print(f"Exported  : {output_path}")
    print(f"Rows      : {n_rows:,}")
    print(f"Size      : {size_mb:.2f} MB")
    print(f"Columns   : {list(df_norm.columns)}")
else:
    print("Nothing to export.")

Exported  : data/raw/csv/en/cybersectony_legit_6606_20260301.csv
Rows      : 6,606
Size      : 29.31 MB
Columns   : ['text', 'label', 'source', 'language']


## 5. Summary

| Criterion | Evidence |
|-----------|----------|
| **Source type** | Fichier de données — Parquet via HuggingFace Hub |
| **Provider** | `cybersectony/PhishingEmailDetectionv2.0` |
| **Content** | English legitimate (non-phishing) emails |
| **Schema** | Normalized to `(text, label, source, language)` |
| **Deduplication** | SHA-256 hash of first 300 chars |

### Role in Pipeline

This legitimate email subset:
1. **Complements** the BigQuery phishing extraction (notebook 03, label=1)
2. **Provides negative examples** for classifier training (ham baseline)
3. **Feeds cultural adaptation** (EN → FR legitimate email patterns)